In [0]:
from pyspark.sql import functions as F

# Read the saved Bronze table
bronze_df = spark.table(
    "dbacademy.get_started_de.bronze_retail"
)

# Profile missing values
bronze_df.select(
    F.count(
        F.when(
            F.col("Description").isNull() |
            (F.trim(F.col("Description")) == ""),
            1
        )
    ).alias("missing_descriptions"),

    F.count(
        F.when(F.col("CustomerID").isNull(), 1)
    ).alias("missing_customer_ids"),

    F.count(
        F.when(F.col("InvoiceNo").isNull(), 1)
    ).alias("missing_invoice_numbers")
).show()

+--------------------+--------------------+-----------------------+
|missing_descriptions|missing_customer_ids|missing_invoice_numbers|
+--------------------+--------------------+-----------------------+
|                1454|              135080|                      0|
+--------------------+--------------------+-----------------------+



In [0]:
# Find exact duplicate rows across all eight columns
duplicate_groups = (
    bronze_df
    .groupBy(*bronze_df.columns)
    .count()
    .filter(F.col("count") > 1)
)

# Count duplicate groups and extra repeated rows
duplicate_groups.agg(
    F.count("*").alias("duplicate_groups"),
    F.sum(F.col("count") - 1).alias("extra_duplicate_rows")
).show()

+----------------+--------------------+
|duplicate_groups|extra_duplicate_rows|
+----------------+--------------------+
|            4879|                5268|
+----------------+--------------------+



Stock Code Validation

In [0]:
from pyspark.sql import functions as F

bronze_df = spark.table(
    "dbacademy.get_started_de.bronze_retail"
)

missing_desc = bronze_df.filter(
    F.col("Description").isNull() |
    (F.trim(F.col("Description")) == "")
)

missing_desc.select(
    "InvoiceNo",
    "StockCode",
    "Description",
    "Quantity",
    "UnitPrice"
).show(20, truncate=False)

missing_desc.agg(
    F.count("*").alias("missing_description_rows"),
    F.countDistinct("StockCode").alias("distinct_product_codes"),
    F.count(
        F.when(F.col("StockCode").isNull(), 1)
    ).alias("missing_product_codes")
).show()

+---------+---------+-----------+--------+---------+
|InvoiceNo|StockCode|Description|Quantity|UnitPrice|
+---------+---------+-----------+--------+---------+
|536414   |22139    |NULL       |56      |0.0      |
|536545   |21134    |NULL       |1       |0.0      |
|536546   |22145    |NULL       |1       |0.0      |
|536547   |37509    |NULL       |1       |0.0      |
|536549   |85226A   |NULL       |1       |0.0      |
|536550   |85044    |NULL       |1       |0.0      |
|536552   |20950    |NULL       |1       |0.0      |
|536553   |37461    |NULL       |3       |0.0      |
|536554   |84670    |NULL       |23      |0.0      |
|536589   |21777    |NULL       |-10     |0.0      |
|536764   |84952C   |NULL       |-38     |0.0      |
|536765   |84952C   |NULL       |19      |0.0      |
|536995   |35951    |NULL       |57      |0.0      |
|536996   |22712    |NULL       |-20     |0.0      |
|536997   |22028    |NULL       |-20     |0.0      |
|536998   |85067    |NULL       |-6      |0.0 

Inconsistent product description

In [0]:
product_variations = (
    bronze_df
    .filter(F.col("Description").isNotNull())
    .groupBy("StockCode")
    .agg(
        F.countDistinct("Description").alias("description_count")
    )
    .filter(F.col("description_count") > 1)
    .orderBy(F.desc("description_count"))
)

display(product_variations.limit(20))

StockCode,description_count
20713,8
23084,7
85175,6
21830,6
72807A,5
85172,5
23343,5
23131,5
21181,5
22121,4


Check same StockCode has a non-missing description


In [0]:
from pyspark.sql import functions as F

bronze_df = spark.table(
    "dbacademy.get_started_de.bronze_retail"
)

# Build a product-code-to-description profile
description_lookup = (
    bronze_df
    .filter(
        F.col("Description").isNotNull() &
        (F.trim(F.col("Description")) != "")
    )
    .groupBy("StockCode")
    .agg(
        F.countDistinct(
            F.trim(F.col("Description"))
        ).alias("distinct_descriptions"),
        F.collect_set(
            F.trim(F.col("Description"))
        ).alias("descriptions")
    )
)

# Join the profile to rows with missing descriptions
missing_description_analysis = (
    bronze_df
    .filter(
        F.col("Description").isNull() |
        (F.trim(F.col("Description")) == "")
    )
    .join(description_lookup, on="StockCode", how="left")
)

display(
    missing_description_analysis.select(
        "InvoiceNo",
        "StockCode",
        "Quantity",
        "UnitPrice",
        "distinct_descriptions",
        "descriptions"
    ).limit(20)
)

InvoiceNo,StockCode,Quantity,UnitPrice,distinct_descriptions,descriptions
536414,22139,56,0.0,2,"List(RETROSPOT TEA SET CERAMIC 11 PC, amazon)"
536545,21134,1,0.0,null,null
536546,22145,1,0.0,1,List(CHRISTMAS CRAFT HEART STOCKING)
536547,37509,1,0.0,1,List(NEW ENGLAND MUG W GIFT BOX)
536549,85226A,1,0.0,null,null
536550,85044,1,0.0,null,null
536552,20950,1,0.0,null,null
536553,37461,3,0.0,1,List(FUNKY MONKEY MUG)
536554,84670,23,0.0,null,null
536589,21777,-10,0.0,1,List(RECIPE BOX WITH METAL HEART)


Count Missing description records

In [0]:
from pyspark.sql import functions as F

missing_description_analysis.groupBy(
    F.when(
        F.col("distinct_descriptions").isNull(),
        "No known description"
    ).when(
        F.col("distinct_descriptions") == 1,
        "One known description"
    ).otherwise(
        "Multiple known descriptions"
    ).alias("recovery_category")
).count().orderBy("recovery_category").show()

+--------------------+-----+
|   recovery_category|count|
+--------------------+-----+
|Multiple known de...|  309|
|No known description|  112|
|One known descrip...| 1033|
+--------------------+-----+



Silver DataFrame

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType

bronze_df = spark.table(
    "dbacademy.get_started_de.bronze_retail"
)

# Build a lookup of distinct, non-empty descriptions per product
description_lookup = (
    bronze_df
    .filter(
        F.col("Description").isNotNull() &
        (F.trim(F.col("Description")) != "")
    )
    .withColumn(
        "clean_description",
        F.trim(F.col("Description"))
    )
    .groupBy("StockCode")
    .agg(
        F.countDistinct("clean_description")
         .alias("description_count"),
        F.first("clean_description")
         .alias("candidate_description")
    )
)

# Clean and standardise the source fields
silver_df = (
    bronze_df
    .withColumn("invoice_no", F.trim("InvoiceNo"))
    .withColumn("product_code", F.trim("StockCode"))
    .withColumn(
        "product_description",
        F.when(
            F.trim("Description") == "",
            None
        ).otherwise(F.trim("Description"))
    )
    .withColumn("quantity", F.col("Quantity").cast("int"))
    .withColumn(
        "invoice_timestamp",
        F.to_timestamp("InvoiceDate", "dd-MM-yyyy HH:mm")
    )
    .withColumn(
        "unit_price",
        F.col("UnitPrice").cast(DecimalType(12, 4))
    )
    .withColumn("customer_id", F.col("CustomerID").cast("string"))
    .withColumn("country", F.trim("Country"))
    .join(description_lookup, on="StockCode", how="left")
    .withColumn(
        "product_description",
        F.when(
            F.col("product_description").isNull() &
            (F.col("description_count") == 1),
            F.col("candidate_description")
        ).otherwise(F.col("product_description"))
    )
    .withColumn(
        "description_status",
        F.when(
            F.col("product_description").isNotNull(),
            "available"
        ).when(
            F.col("description_count") > 1,
            "ambiguous"
        ).otherwise("unresolved")
    )
    .drop("description_count", "candidate_description")
)

Compare Bronze and Silver table to make sure no data is missing

In [0]:
print("Bronze rows:", bronze_df.count())
print("Silver rows:", silver_df.count())

display(
    silver_df.select(
        "invoice_no",
        "product_code",
        "product_description",
        "quantity",
        "invoice_timestamp",
        "unit_price",
        "customer_id",
        "country",
        "description_status"
    ).limit(10)
)

Bronze rows: 541909
Silver rows: 541909


invoice_no,product_code,product_description,quantity,invoice_timestamp,unit_price,customer_id,country,description_status
536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01T08:26:00.000Z,2.5500,17850,United Kingdom,available
536365,71053,WHITE METAL LANTERN,6,2010-12-01T08:26:00.000Z,3.3900,17850,United Kingdom,available
536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01T08:26:00.000Z,2.7500,17850,United Kingdom,available
536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01T08:26:00.000Z,3.3900,17850,United Kingdom,available
536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01T08:26:00.000Z,3.3900,17850,United Kingdom,available
536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01T08:26:00.000Z,7.6500,17850,United Kingdom,available
536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01T08:26:00.000Z,4.2500,17850,United Kingdom,available
536366,22633,HAND WARMER UNION JACK,6,2010-12-01T08:28:00.000Z,1.8500,17850,United Kingdom,available
536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01T08:28:00.000Z,1.8500,17850,United Kingdom,available
536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,2010-12-01T08:34:00.000Z,1.6900,13047,United Kingdom,available


In [0]:
silver_df.groupBy(
    "description_status"
).count().show()

+------------------+------+
|description_status| count|
+------------------+------+
|         ambiguous|   309|
|         available|541488|
|        unresolved|   112|
+------------------+------+



In [0]:
silver_df.filter(
    F.col("InvoiceDate").isNotNull() &
    F.col("invoice_timestamp").isNull()
).select(
    "InvoiceDate"
).show(20, truncate=False)

+-----------+
|InvoiceDate|
+-----------+
+-----------+



Data Quality Check

In [0]:
silver_df.select(
    F.count(
        F.when(F.col("invoice_no").isNull(), 1)
    ).alias("missing_invoice_no"),

    F.count(
        F.when(F.col("product_code").isNull(), 1)
    ).alias("missing_product_code"),

    F.count(
        F.when(F.col("quantity").isNull(), 1)
    ).alias("invalid_quantity"),

    F.count(
        F.when(F.col("invoice_timestamp").isNull(), 1)
    ).alias("invalid_invoice_timestamp"),

    F.count(
        F.when(F.col("unit_price").isNull(), 1)
    ).alias("invalid_unit_price")
).show()

+------------------+--------------------+----------------+-------------------------+------------------+
|missing_invoice_no|missing_product_code|invalid_quantity|invalid_invoice_timestamp|invalid_unit_price|
+------------------+--------------------+----------------+-------------------------+------------------+
|                 0|                   0|               0|                        0|                 0|
+------------------+--------------------+----------------+-------------------------+------------------+



Silver Table Creation

In [0]:
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dbacademy.get_started_de.silver_retail")

In [0]:
silver_table = spark.table(
    "dbacademy.get_started_de.silver_retail"
)

print("Silver rows:", silver_table.count())

display(
    silver_table.select(
        "invoice_no",
        "product_code",
        "product_description",
        "description_status",
        "quantity",
        "unit_price",
        "customer_id",
        "country"
    ).limit(10)
)

Silver rows: 541909


invoice_no,product_code,product_description,description_status,quantity,unit_price,customer_id,country
536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,available,6,2.5500,17850,United Kingdom
536365,71053,WHITE METAL LANTERN,available,6,3.3900,17850,United Kingdom
536365,84406B,CREAM CUPID HEARTS COAT HANGER,available,8,2.7500,17850,United Kingdom
536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,available,6,3.3900,17850,United Kingdom
536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,available,6,3.3900,17850,United Kingdom
536365,22752,SET 7 BABUSHKA NESTING BOXES,available,2,7.6500,17850,United Kingdom
536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,available,6,4.2500,17850,United Kingdom
536366,22633,HAND WARMER UNION JACK,available,6,1.8500,17850,United Kingdom
536366,22632,HAND WARMER RED POLKA DOT,available,6,1.8500,17850,United Kingdom
536367,84879,ASSORTED COLOUR BIRD ORNAMENT,available,32,1.6900,13047,United Kingdom


In [0]:
from pyspark.sql import functions as F

bronze_df = spark.table(
    "dbacademy.get_started_de.bronze_retail"
)

# Find repeated rows across all original source columns
duplicate_groups = (
    bronze_df
    .groupBy(*bronze_df.columns)
    .count()
    .filter(F.col("count") > 1)
)

# Estimate the impact of extra copies
duplicate_impact = duplicate_groups.agg(
    F.sum(F.col("count") - 1).alias("extra_rows"),

    F.sum(
        (F.col("count") - 1) * F.col("Quantity")
    ).alias("extra_quantity"),

    F.sum(
        (F.col("count") - 1) *
        F.col("Quantity") *
        F.col("UnitPrice")
    ).alias("potential_extra_sales_value")
)

display(duplicate_impact)

extra_rows,extra_quantity,potential_extra_sales_value
5268,13948,21740.9800000003


In [0]:
from pyspark.sql import functions as F

display(
    duplicate_groups
    .orderBy(F.desc("count"))
    .limit(20)
)

InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,count
555524,22698,PINK REGENCY TEACUP AND SAUCER,1,05-06-2011 11:37,2.95,16923,United Kingdom,20
555524,22697,GREEN REGENCY TEACUP AND SAUCER,1,05-06-2011 11:37,2.95,16923,United Kingdom,12
572861,22775,PURPLE DRAWERKNOB ACRYLIC EDWARDIAN,12,26-10-2011 12:46,1.25,14102,United Kingdom,8
538514,21756,BATH BUILDING BLOCK WORD,1,12-12-2010 14:27,5.95,15044,United Kingdom,6
578289,23395,BELLE JARDINIERE CUSHION COVER,1,23-11-2011 14:07,3.75,17841,United Kingdom,6
572344,M,Manual,48,24-10-2011 10:43,1.5,14607,United Kingdom,6
541266,21755,LOVE BUILDING BLOCK WORD,1,16-01-2011 16:25,5.95,15673,United Kingdom,6
541266,21754,HOME BUILDING BLOCK WORD,1,16-01-2011 16:25,5.95,15673,United Kingdom,6
540524,21756,BATH BUILDING BLOCK WORD,1,09-01-2011 12:53,5.95,16735,United Kingdom,6
550331,85202,HANGING WOOD AND FELT HEART,1,17-04-2011 13:43,0.42,15555,United Kingdom,5


Duplicate Investigation Dataset

In [0]:
from pyspark.sql import functions as F

duplicate_candidates = (
    duplicate_groups
    .withColumn(
        "extra_rows",
        F.col("count") - 1
    )
    .withColumn(
        "potential_extra_value",
        F.col("extra_rows") *
        F.col("Quantity") *
        F.col("UnitPrice")
    )
    .select(
        *bronze_df.columns,
        F.col("count").alias("occurrence_count"),
        "extra_rows",
        "potential_extra_value"
    )
    .orderBy(F.desc("extra_rows"))
)

display(duplicate_candidates.limit(20))

InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,occurrence_count,extra_rows,potential_extra_value
555524,22698,PINK REGENCY TEACUP AND SAUCER,1,05-06-2011 11:37,2.95,16923,United Kingdom,20,19,56.050000000000004
555524,22697,GREEN REGENCY TEACUP AND SAUCER,1,05-06-2011 11:37,2.95,16923,United Kingdom,12,11,32.45
572861,22775,PURPLE DRAWERKNOB ACRYLIC EDWARDIAN,12,26-10-2011 12:46,1.25,14102,United Kingdom,8,7,105.0
538514,21756,BATH BUILDING BLOCK WORD,1,12-12-2010 14:27,5.95,15044,United Kingdom,6,5,29.75
578289,23395,BELLE JARDINIERE CUSHION COVER,1,23-11-2011 14:07,3.75,17841,United Kingdom,6,5,18.75
572344,M,Manual,48,24-10-2011 10:43,1.5,14607,United Kingdom,6,5,360.0
541266,21755,LOVE BUILDING BLOCK WORD,1,16-01-2011 16:25,5.95,15673,United Kingdom,6,5,29.75
541266,21754,HOME BUILDING BLOCK WORD,1,16-01-2011 16:25,5.95,15673,United Kingdom,6,5,29.75
540524,21756,BATH BUILDING BLOCK WORD,1,09-01-2011 12:53,5.95,16735,United Kingdom,6,5,29.75
550331,85202,HANGING WOOD AND FELT HEART,1,17-04-2011 13:43,0.42,15555,United Kingdom,5,4,1.68


In [0]:
display(
    duplicate_candidates
    .orderBy(
        F.desc("occurrence_count"),
        F.desc("potential_extra_value")
    )
    .limit(20)
)

InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,occurrence_count,extra_rows,potential_extra_value
555524,22698,PINK REGENCY TEACUP AND SAUCER,1,05-06-2011 11:37,2.95,16923,United Kingdom,20,19,56.050000000000004
555524,22697,GREEN REGENCY TEACUP AND SAUCER,1,05-06-2011 11:37,2.95,16923,United Kingdom,12,11,32.45
572861,22775,PURPLE DRAWERKNOB ACRYLIC EDWARDIAN,12,26-10-2011 12:46,1.25,14102,United Kingdom,8,7,105.0
572344,M,Manual,48,24-10-2011 10:43,1.5,14607,United Kingdom,6,5,360.0
541266,21754,HOME BUILDING BLOCK WORD,1,16-01-2011 16:25,5.95,15673,United Kingdom,6,5,29.75
541266,21755,LOVE BUILDING BLOCK WORD,1,16-01-2011 16:25,5.95,15673,United Kingdom,6,5,29.75
540524,21756,BATH BUILDING BLOCK WORD,1,09-01-2011 12:53,5.95,16735,United Kingdom,6,5,29.75
538514,21756,BATH BUILDING BLOCK WORD,1,12-12-2010 14:27,5.95,15044,United Kingdom,6,5,29.75
578289,23395,BELLE JARDINIERE CUSHION COVER,1,23-11-2011 14:07,3.75,17841,United Kingdom,6,5,18.75
538517,22749,FELTCRAFT PRINCESS CHARLOTTE DOLL,1,12-12-2010 16:05,3.75,17618,United Kingdom,5,4,15.0


In [0]:
duplicate_candidates.groupBy(
    "occurrence_count"
).agg(
    F.count("*").alias("duplicate_groups"),
    F.sum("extra_rows").alias("extra_rows")
).orderBy("occurrence_count").show()

+----------------+----------------+----------+
|occurrence_count|duplicate_groups|extra_rows|
+----------------+----------------+----------+
|               2|            4588|      4588|
|               3|             238|       476|
|               4|              39|       117|
|               5|               5|        20|
|               6|               6|        30|
|               8|               1|         7|
|              12|               1|        11|
|              20|               1|        19|
+----------------+----------------+----------+



Investigate the duplicate records

In [0]:
display(
    duplicate_candidates
    .filter(F.col("occurrence_count") >= 8)
    .select(
        "InvoiceNo",
        "StockCode",
        "Description",
        "Quantity",
        "InvoiceDate",
        "UnitPrice",
        "CustomerID",
        "Country",
        "occurrence_count",
        "extra_rows",
        "potential_extra_value"
    )
    .orderBy(F.desc("occurrence_count"))
)

InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,occurrence_count,extra_rows,potential_extra_value
555524,22698,PINK REGENCY TEACUP AND SAUCER,1,05-06-2011 11:37,2.95,16923,United Kingdom,20,19,56.050000000000004
555524,22697,GREEN REGENCY TEACUP AND SAUCER,1,05-06-2011 11:37,2.95,16923,United Kingdom,12,11,32.45
572861,22775,PURPLE DRAWERKNOB ACRYLIC EDWARDIAN,12,26-10-2011 12:46,1.25,14102,United Kingdom,8,7,105.0


In [0]:
duplicate_candidates.filter(
    F.col("occurrence_count") >= 8
).select(
    "occurrence_count",
    "InvoiceNo",
    "StockCode",
    "Description",
    "extra_rows"
).orderBy(
    F.desc("occurrence_count")
).show(20, truncate=False)

+----------------+---------+---------+-----------------------------------+----------+
|occurrence_count|InvoiceNo|StockCode|Description                        |extra_rows|
+----------------+---------+---------+-----------------------------------+----------+
|20              |555524   |22698    |PINK REGENCY TEACUP AND SAUCER     |19        |
|12              |555524   |22697    |GREEN REGENCY TEACUP AND SAUCER    |11        |
|8               |572861   |22775    |PURPLE DRAWERKNOB ACRYLIC EDWARDIAN|7         |
+----------------+---------+---------+-----------------------------------+----------+



In [0]:
duplicate_candidates.groupBy(
    "occurrence_count"
).count().orderBy(
    F.desc("occurrence_count")
).show()

+----------------+-----+
|occurrence_count|count|
+----------------+-----+
|              20|    1|
|              12|    1|
|               8|    1|
|               6|    6|
|               5|    5|
|               4|   39|
|               3|  238|
|               2| 4588|
+----------------+-----+



In [0]:
duplicate_candidates.filter(
    F.col("occurrence_count") >= 8
).select(
    "InvoiceNo",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "UnitPrice",
    "CustomerID",
    "Country",
    "occurrence_count",
    "extra_rows",
    "potential_extra_value"
).orderBy(
    F.desc("occurrence_count")
).show(10, truncate=False)

+---------+---------+-----------------------------------+--------+----------------+---------+----------+--------------+----------------+----------+---------------------+
|InvoiceNo|StockCode|Description                        |Quantity|InvoiceDate     |UnitPrice|CustomerID|Country       |occurrence_count|extra_rows|potential_extra_value|
+---------+---------+-----------------------------------+--------+----------------+---------+----------+--------------+----------------+----------+---------------------+
|555524   |22698    |PINK REGENCY TEACUP AND SAUCER     |1       |05-06-2011 11:37|2.95     |16923     |United Kingdom|20              |19        |56.050000000000004   |
|555524   |22697    |GREEN REGENCY TEACUP AND SAUCER    |1       |05-06-2011 11:37|2.95     |16923     |United Kingdom|12              |11        |32.45                |
|572861   |22775    |PURPLE DRAWERKNOB ACRYLIC EDWARDIAN|12      |26-10-2011 12:46|1.25     |14102     |United Kingdom|8               |7         |105

Investigating the most repeated group in Bronze Table

In [0]:
bronze_df.filter(
    (F.col("InvoiceNo") == "555524") &
    (F.col("StockCode") == "22698")
).groupBy(
    *bronze_df.columns
).count().show(truncate=False)

+---------+---------+------------------------------+--------+----------------+---------+----------+--------------+-----+
|InvoiceNo|StockCode|Description                   |Quantity|InvoiceDate     |UnitPrice|CustomerID|Country       |count|
+---------+---------+------------------------------+--------+----------------+---------+----------+--------------+-----+
|555524   |22698    |PINK REGENCY TEACUP AND SAUCER|1       |05-06-2011 11:37|2.95     |16923     |United Kingdom|20   |
+---------+---------+------------------------------+--------+----------------+---------+----------+--------------+-----+



In [0]:
bronze_df.filter(
    F.col("InvoiceNo") == "555524"
).groupBy(
    "InvoiceNo",
    "StockCode",
    "Description",
    "Quantity",
    "UnitPrice",
    "CustomerID",
    "Country"
).count().orderBy(
    F.desc("count")
).show(50, truncate=False)

+---------+---------+-----------------------------------+--------+---------+----------+--------------+-----+
|InvoiceNo|StockCode|Description                        |Quantity|UnitPrice|CustomerID|Country       |count|
+---------+---------+-----------------------------------+--------+---------+----------+--------------+-----+
|555524   |22698    |PINK REGENCY TEACUP AND SAUCER     |1       |2.95     |16923     |United Kingdom|20   |
|555524   |22697    |GREEN REGENCY TEACUP AND SAUCER    |1       |2.95     |16923     |United Kingdom|12   |
|555524   |23170    |REGENCY TEA PLATE ROSES            |12      |1.65     |16923     |United Kingdom|2    |
|555524   |23173    |REGENCY TEAPOT ROSES               |1       |9.95     |16923     |United Kingdom|2    |
|555524   |21155    |RED RETROSPOT PEG BAG              |1       |2.55     |16923     |United Kingdom|2    |
|555524   |22776    |SWEETHEART CAKESTAND 3 TIER        |1       |9.95     |16923     |United Kingdom|2    |
|555524   |22799   

Scenario Analysis

In [0]:
from pyspark.sql import functions as F

invoice_555524 = bronze_df.filter(
    F.col("InvoiceNo") == "555524"
)

# Current value using every source row
original_value = invoice_555524.agg(
    F.sum(
        F.col("Quantity") * F.col("UnitPrice")
    ).alias("original_invoice_value")
)

# Hypothetical value after retaining one row
# per exact combination of source fields
deduplicated_invoice = invoice_555524.dropDuplicates(
    bronze_df.columns
)

deduplicated_value = deduplicated_invoice.agg(
    F.sum(
        F.col("Quantity") * F.col("UnitPrice")
    ).alias("hypothetical_invoice_value")
)

original_value.show()
deduplicated_value.show()

+----------------------+
|original_invoice_value|
+----------------------+
|     610.7699999999999|
+----------------------+

+--------------------------+
|hypothetical_invoice_value|
+--------------------------+
|         480.0200000000001|
+--------------------------+




Hypothetical Scenario for the full dataset


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType

# Use decimal arithmetic for financial calculations
bronze_financial = bronze_df.withColumn(
    "line_value",
    F.col("Quantity").cast("decimal(18,0)") *
    F.col("UnitPrice").cast(DecimalType(18, 4))
)

# Baseline: all source rows
baseline = bronze_financial.agg(
    F.sum("line_value").alias("original_value"),
    F.count("*").alias("original_rows")
)

# Scenario: retain one copy of each exact source row
deduplicated_financial = (
    bronze_financial
    .dropDuplicates(bronze_df.columns)
)

scenario = deduplicated_financial.agg(
    F.sum("line_value").alias("scenario_value"),
    F.count("*").alias("scenario_rows")
)

baseline.show()
scenario.show()

+--------------+-------------+
|original_value|original_rows|
+--------------+-------------+
|  9747747.9340|       541909|
+--------------+-------------+

+--------------+-------------+
|scenario_value|scenario_rows|
+--------------+-------------+
|  9726006.9540|       536641|
+--------------+-------------+



In [0]:
from pyspark.sql import functions as F

silver_table = spark.table(
    "dbacademy.get_started_de.silver_retail"
)

silver_table.select(
    F.count("*").alias("total_rows"),

    F.count(
        F.when(
            F.col("invoice_no").startswith("C"),
            1
        )
    ).alias("cancellation_invoice_rows"),

    F.count(
        F.when(F.col("quantity") < 0, 1)
    ).alias("negative_quantity_rows"),

    F.count(
        F.when(F.col("unit_price") == 0, 1)
    ).alias("zero_price_rows")
).show()

+----------+-------------------------+----------------------+---------------+
|total_rows|cancellation_invoice_rows|negative_quantity_rows|zero_price_rows|
+----------+-------------------------+----------------------+---------------+
|    541909|                     9288|                 10624|           2515|
+----------+-------------------------+----------------------+---------------+



Data Classification

In [0]:
from pyspark.sql import functions as F

silver_table = spark.table(
    "dbacademy.get_started_de.silver_retail"
)

classified = silver_table.withColumn(
    "is_cancellation_invoice",
    F.upper(F.trim(F.col("invoice_no"))).startswith("C")
).withColumn(
    "is_negative_quantity",
    F.col("quantity") < 0
).withColumn(
    "is_zero_price",
    F.col("unit_price") == 0
)

display(
    classified.groupBy(
        "is_cancellation_invoice",
        "is_negative_quantity",
        "is_zero_price"
    ).agg(
        F.count("*").alias("row_count")
    ).orderBy(
        F.desc("row_count")
    )
)

is_cancellation_invoice,is_negative_quantity,is_zero_price,row_count
false,false,false,530106
true,true,false,9288
false,true,true,1336
false,false,true,1179


In [0]:
from pyspark.sql import functions as F

silver_table = spark.table(
    "dbacademy.get_started_de.silver_retail"
)

silver_table.agg(
    F.sum(
        F.col("quantity").cast("decimal(18,0)") *
        F.col("unit_price").cast("decimal(18,4)")
    ).alias("signed_transaction_value"),

    F.sum(
        F.when(
            F.col("quantity") > 0,
            F.col("quantity").cast("decimal(18,0)") *
            F.col("unit_price").cast("decimal(18,4)")
        ).otherwise(F.lit(0).cast("decimal(30,4)"))
    ).alias("positive_quantity_value"),

    F.sum(
        F.when(
            F.col("quantity") < 0,
            F.col("quantity").cast("decimal(18,0)") *
            F.col("unit_price").cast("decimal(18,4)")
        ).otherwise(F.lit(0).cast("decimal(30,4)"))
    ).alias("negative_quantity_value")
).show()

+------------------------+-----------------------+-----------------------+
|signed_transaction_value|positive_quantity_value|negative_quantity_value|
+------------------------+-----------------------+-----------------------+
|            9747747.9340|          10644560.4240|           -896812.4900|
+------------------------+-----------------------+-----------------------+

